## TODO:

- Advanced Messages Operations in LangChain: trim_messages, filter_messages, count_tokens_approximately: https://reference.langchain.com/python/langchain-core/messages/utils
- Look for Multimodal Messages as well

Till now we have been providing only text as an input to the LLMs. Let's now learn how to provide images and audio as well to these LLMs. This will give the ability so that they can see and hear the world around them.

First things first, we will be encoding our image and audio files in Base64. For reference, binary is Base2. 

Base64 officially converts binary files into long text strings.

This enables us to transmit images and audios on the text-based communication channel like HTTP or inside JSON. Note that JSON cannot contain raw binary.

**Important:**
- Binary transport allowed? → Do NOT use base64.
- Text-only transport?      → Use base64.
- As it increases memory usage and adds encoding/decoding overhead.

In [2]:
import base64
from pathlib import Path


def image_to_base64(image_path: str) -> str:
    """
    Load an image from disk and convert it to a Base64-encoded string.

    Args:
        image_path: Path to the image file.

    Returns:
        Base64-encoded string representation of the image.
    """
    path = Path(image_path)

    if not path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    with path.open("rb") as f:
        encoded_bytes = base64.b64encode(f.read()).decode("utf-8")

    return encoded_bytes

In [ ]:
encoded_image = image_to_base64("example.jpg")
print(encoded_image[:100])  # print first 100 characters

In [ ]:
from langchain.messages import HumanMessage

multimodal_query = HumanMessage(content=[
    {"type" : "text", "text" : "Explain what you see."},
    {"type" : "image", "mime_type" : "image/png", "base64" : encoded_image}
])

In [ ]:
response = agent.invoke(
    {"messages" : [multimodal_query]}
)

In [ ]:
print(response["messages"][-1].content)

In [ ]:
import librosa

def load_and_resample(path, target_sr=16000):
    audio, sr = librosa.load(path, sr=None)

    if sr != target_sr:
        audio = librosa.resample(
            y=audio,
            orig_sr=sr,
            target_sr=target_sr
        )

    return audio

In [ ]:
# classifier.feature_extractor.sampling_rate

audio = load_and_resample("esc50_samples/sample_0.wav", target_sr=16,000)



In [ ]:
multimodal_query = HumanMessage(content=[
    {"type" : "text", "text" : "Explain what you see."},
    {"type" : "audio", "mime_type" : "audio/wav", "base64" : audio_b64}
])